In [4]:
'''Loading GCA dataset'''

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import Lasso
from sklearn.svm import SVC, SVR
from sklearn.feature_selection import RFE, SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from imblearn.over_sampling import SMOTE
from skrebate import ReliefF  # pip install skrebate — real ReliefF
from itertools import combinations
import csv
df = pd.read_csv('../data/GSE201753/normalized_data_deg.csv')
print(df.shape)


labels_df = pd.read_csv("../data/GSE201753/GSE201753_metadata.csv", index_col=0)
labels = labels_df['condition']

first_col_name = df.columns[0]
first_col = df.iloc[:, 0]

# 1. Isolate the numeric data first
df_numeric = df.iloc[:, 1:]

# 2. Calculate variance ONLY on the numeric columns
gene_var = df_numeric.var(axis=0)
low_var_threshold = gene_var.quantile(0.20)  # Bottom 20% variance threshold

# 3. Filter the numeric columns using the correctly aligned mask
df_filtered = df_numeric.loc[:, gene_var >= low_var_threshold]

# 4. Insert the first column back at the beginning
df_filtered.insert(0, first_col_name, first_col)
df = df_filtered

print(f"Data shape after filtering low-variance genes further: {df.shape}  (samples × genes)")

df.to_csv('normalized_and_filtered_data_deg.csv', index=False)

df.head()




(109, 3683)
Data shape after filtering low-variance genes further: (109, 2946)  (samples × genes)


/var/folders/mk/dhch81x172qcjg2g_ztdf3qw0000gn/T/ipykernel_41195/1226909579.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_filtered.insert(0, first_col_name, first_col)


,condition,LOC729737,LOC100132287,LINC01128,FAM41C,KLHL17,PLEKHN1,HES4,AGRN,LOC100288175,...,UBL4A,SLC10A3,SNORA56,MPP1,LOC283788,LOC100505874,LOC112268317,MT-TM,MT-TA,MT-TK
0,RNT,11.578113,10.389591,9.497614,0.000000,7.073410,6.848461,7.498750,8.651213,6.525859,...,10.392203,10.804686,3.051673,12.330065,9.782783,0.000000,5.361224,11.391666,11.075570,10.633146
1,RNT,9.929359,9.615416,8.479225,3.050900,6.580971,5.595962,6.230500,8.054043,4.514871,...,10.054916,10.443847,4.514871,12.422790,9.610941,1.777828,3.843987,10.856737,12.043144,9.964859
2,RT,8.793945,8.279421,7.812752,2.280750,5.730656,5.159215,4.410161,6.358350,4.199034,...,10.078958,10.198682,2.954900,12.152311,10.150509,2.542042,8.088078,9.778121,12.138794,9.035900
3,RT,11.265795,10.493648,7.880989,2.122103,8.220737,6.348351,6.682519,7.445038,5.109895,...,9.650247,9.751000,0.000000,12.681448,10.410784,0.000000,2.720015,11.668848,14.107598,11.125597
4,RT,10.827181,9.808508,8.063747,2.405840,7.270030,5.755593,5.411468,7.493001,5.550306,...,9.727902,10.000375,1.282823,12.538672,11.018731,3.463626,5.676948,12.399419,13.433328,11.329515


In [ ]:
'''Feature selection for the top 100 genes'''

def select_features_on_fold(X_train, y_train, n_features=100):
    """
    All feature selection runs on training fold ONLY.
    Fixes data leakage completely.
    """
        

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_train)  # fit on train only
    gene_stds = X_scaled.std(axis=0)
    feature_names = X_train.columns

    print(f"  Genes before near-zero variance filter: {X_scaled.shape[1]}")
    valid_genes = gene_stds > 0.01          # drop near-constant genes
    X_scaled    = X_scaled[:, valid_genes]
    feature_names = feature_names[valid_genes]
    print(f"  Genes after near-zero variance filter: {X_scaled.shape[1]}")

    selected = {}

    # ── 1. Lasso ─────────────────────────────────────────────
    # Use LassoCV or tune alpha — alpha=0.01 may be too permissive
    lasso = Lasso(alpha=0.01, max_iter=5000)
    lasso.fit(X_scaled, y_train)
    coef_mask = np.abs(lasso.coef_) > 0
    if coef_mask.sum() < 5:
        # Fallback: take top n by coefficient magnitude if too few selected
        idx = np.argsort(np.abs(lasso.coef_))[::-1][:n_features]
    else:
        idx = np.where(coef_mask)[0][:n_features]
    selected['Lasso'] = set(feature_names[idx])

    # ── 2. SVM-RFE ───────────────────────────────────────────
    svm_linear = SVC(kernel='linear', class_weight='balanced')
    rfe = RFE(estimator=svm_linear, n_features_to_select=n_features, step=0.1)
    rfe.fit(X_scaled, y_train)
    selected['SVM-RFE'] = set(feature_names[rfe.support_])

    # ── 3. Random Forest ─────────────────────────────────────
    rf = RandomForestClassifier(
        n_estimators=500,         # more trees = more stable importances
        max_features='sqrt',
        min_samples_leaf=2,       # slight regularization
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_scaled, y_train)
    rf_idx = np.argsort(rf.feature_importances_)[::-1][:n_features]
    selected['Random Forest'] = set(feature_names[rf_idx])

   # ── 4. SVM-RBF — replace permutation importance entirely ──────────────
    # Strategy: use an RBF SVM to score samples, then correlate each gene
    # with the decision confidence. Genes that most strongly drive confident
    # correct predictions are ranked highest.

    svm_rbf = SVC(kernel='rbf', class_weight='balanced', probability=True)
    svm_rbf.fit(X_scaled, y_train)

    # Get probability predictions (confidence per class per sample)
    proba = svm_rbf.predict_proba(X_scaled)           # shape: (n_samples, n_classes)
    max_confidence = proba.max(axis=1)                 # how confident the model is per sample

    # Rank genes by their correlation with prediction confidence
    # Genes that vary in step with confident predictions are informative
    rbf_scores = np.array([
        abs(np.corrcoef(X_scaled[:, i], max_confidence)[0, 1])
        for i in range(X_scaled.shape[1])
    ])
    rbf_scores = np.nan_to_num(rbf_scores)             # replace NaN with 0
    rbf_idx = np.argsort(rbf_scores)[::-1][:n_features]
    selected['SVM-RBF'] = set(feature_names[rbf_idx])

    # ── 5. Double-RBF — two-stage RBF filtering ───────────────────────────
    # Stage 1: keep top 50% genes by RBF score above
    stage1_mask = np.argsort(rbf_scores)[::-1][:X_scaled.shape[1] // 2]
    X_stage1    = X_scaled[:, stage1_mask]
    names_stage1 = feature_names[stage1_mask]

    # Stage 2: fit a new RBF SVM on the reduced gene set, re-rank
    svm_rbf2 = SVC(kernel='rbf', class_weight='balanced', probability=True)
    svm_rbf2.fit(X_stage1, y_train)
    proba2        = svm_rbf2.predict_proba(X_stage1)
    confidence2   = proba2.max(axis=1)

    rbf2_scores = np.array([
        abs(np.corrcoef(X_stage1[:, i], confidence2)[0, 1])
        for i in range(X_stage1.shape[1])
    ])
    rbf2_scores = np.nan_to_num(rbf2_scores)
    # After computing rbf2_scores, map back to full feature space
    rbf2_scores_full = np.zeros(len(feature_names))        # start with zeros
    rbf2_scores_full[stage1_mask] = rbf2_scores            # fill in stage-1 positions
    stage2_local_idx  = np.argsort(rbf2_scores)[::-1][:n_features]
    selected['Double-RBF'] = set(names_stage1[stage2_local_idx])

    # ── 6. ReliefF (REAL implementation via skrebate) ────────
    # ReliefF works on raw (non-encoded) multiclass labels
    relief = ReliefF(n_features_to_select=n_features, n_neighbors=10)
    relief.fit(X_scaled, y_train)
    relief_idx = np.argsort(relief.feature_importances_)[::-1][:n_features]
    selected['ReliefF'] = set(feature_names[relief_idx])

    # ── 7. SigFeature (SVM weights × t-stat) ─────────────────
    ovr = OneVsRestClassifier(SVC(kernel='linear', class_weight='balanced'))
    ovr.fit(X_scaled, y_train)
    coefficients = np.vstack([est.coef_ for est in ovr.estimators_])
    svm_weights = np.abs(coefficients).mean(axis=0)

    # t-statistic across all class pairs
    classes_unique = np.unique(y_train)
    t_stats = np.zeros(X_scaled.shape[1])
    for i in range(X_scaled.shape[1]):
        pair_stats = []
        for c1, c2 in combinations(classes_unique, 2):
            g1 = X_scaled[y_train == c1, i]
            g2 = X_scaled[y_train == c2, i]
            if len(g1) > 1 and len(g2) > 1:
                from scipy.stats import ttest_ind
                t, _ = ttest_ind(g1, g2)
                pair_stats.append(abs(t))
        t_stats[i] = np.mean(pair_stats) if pair_stats else 0

    sig_scores = svm_weights * t_stats
    sig_idx = np.argsort(sig_scores)[::-1][:n_features]
    selected['SigFeature'] = set(feature_names[sig_idx])

    # ── 8. Ensemble (rank aggregation — Borda count) ──────────
    # More robust than averaging raw importances across different scales
    all_importances = {
        'Lasso':       np.abs(lasso.coef_),
        'RF':          rf.feature_importances_,
        'SVM-RFE':     rfe.ranking_ * -1,  # lower rank = better, so negate
        'SVM-RBF':     rbf_scores,
        'Double-RBF':  rbf2_scores_full,
        'SigFeature':  sig_scores,
        'ReliefF':     relief.feature_importances_,
    }

    borda_scores = np.zeros(len(feature_names))
    for name, scores in all_importances.items():
        ranks = np.argsort(np.argsort(scores))  # rank 0 = worst
        borda_scores += ranks

    ensemble_idx = np.argsort(borda_scores)[::-1][:n_features]
    selected['Ensemble'] = set(feature_names[ensemble_idx])

    return selected, scaler


In [10]:
'''Evaluation metrics'''
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)
from sklearn.preprocessing import label_binarize

def compute_metrics(y_true, y_pred, y_proba=None, n_classes=None):
    results = {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'confusion_matrix': confusion_matrix(y_true, y_pred),
    }

    if y_proba is not None and n_classes is not None:
        y_true_bin = label_binarize(y_true, classes=np.arange(n_classes))
        results['roc_auc_ovr'] = roc_auc_score(y_true_bin, y_proba, average='macro', multi_class='ovr')
        results['pr_auc_macro'] = average_precision_score(y_true_bin, y_proba, average='macro')

    return results

In [ ]:
'''Splitting the data and training the models'''
from collections import defaultdict

from sklearn.model_selection import train_test_split

target_col = 'condition'
X_all = df.drop(columns=[target_col])
y_all = df[target_col].values
y_enc = LabelEncoder().fit_transform(y_all)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_all,
    y_enc,
    test_size=0.2,
    stratify=y_enc,
    random_state=42
)

#Models to evaluate (change max_iter and n_estimators as needed)
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced', C=0.1),
    'Random Forest': RandomForestClassifier(n_estimators=3000, class_weight='balanced', max_features='sqrt', random_state=42),
    'SVM-Linear': SVC(kernel='linear', class_weight='balanced', C=0.1),
}

#Selects features and applies SMOTE
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

#Makes sure that each of the 4 conditions is represented in each fold
n_classes = len(np.unique(y_enc))
results_by_method = defaultdict(list)

for fold, (train_idx, val_idx) in enumerate(cv.split(X_train_full, y_train_full), start=1):
    X_tr = X_train_full.iloc[train_idx]
    X_val = X_train_full.iloc[val_idx]
    y_tr = y_train_full[train_idx]
    y_val = y_train_full[val_idx]

    # SMOTE only on train fold
    X_tr_res, y_tr_res = SMOTE(random_state=42, k_neighbors=3).fit_resample(X_tr, y_tr)
    X_tr_res = pd.DataFrame(X_tr_res, columns=X_tr.columns)

    # Feature selection only on train fold
    selected_dict, scaler = select_features_on_fold(X_tr_res, y_tr_res)

    #Results list to store metrics for each classifier and feature selection method
    
    for method, genes in selected_dict.items():
        X_tr_sel = X_tr_res[list(genes)]
        X_val_sel = X_val[list(genes)]

        fold_scaler = StandardScaler().fit(X_tr_sel)
        X_tr_sel = fold_scaler.transform(X_tr_sel)
        X_val_sel = fold_scaler.transform(X_val_sel)
        
        #Train the model on this fold
        current_clf_results = defaultdict(list)  # Store results for this classifier and method
        for clf_name, clf in classifiers.items():
            
            clf.fit(X_tr_sel, y_tr_res)

            y_tr_pred = clf.predict(X_tr_sel)
            y_val_pred = clf.predict(X_val_sel)

            y_tr_proba = clf.predict_proba(X_tr_sel) if hasattr(clf, "predict_proba") else None
            y_val_proba = clf.predict_proba(X_val_sel) if hasattr(clf, "predict_proba") else None

            train_metrics = compute_metrics(y_tr_res, y_tr_pred, y_proba=y_tr_proba, n_classes=n_classes)
            val_metrics = compute_metrics(y_val, y_val_pred, y_proba=y_val_proba, n_classes=n_classes)

            print(f"Fold {fold} | {method} | {clf_name}")
            print(f"  Train acc: {train_metrics['accuracy']:.4f}, F1: {train_metrics['f1_macro']:.4f}")
            print(f"  Val   acc: {val_metrics['accuracy']:.4f}, F1: {val_metrics['f1_macro']:.4f}")
            
            current_clf_results[clf_name].append((train_metrics, val_metrics))
            
            #After the method and classifier have been evaluated, store the results
            results_by_method[method].append({
                'Fold': fold,
                'Classifier': clf_name,
                'Accuracy': train_metrics['accuracy'],
                'F1 Score (train)': train_metrics['f1_macro'], 
                'F1 Score (val)': val_metrics['f1_macro'],
                'Num_Features': len(genes),
                'Features': '; '.join(sorted(list(genes))),
                })
#Writing results to CSV after each fold to avoid data loss
for method, results in results_by_method.items():
    pd.DataFrame(results).to_csv(f"top_hundred_results/results_{method}.csv", index=False)
    print(f"Saved results for {method}\n")
    

  Genes before near-zero variance filter: 2945
  Genes after near-zero variance filter: 2945


/Users/damianmoreno/Documents/DePaul_Cobab/GCA_Project/.venv/lib/python3.13/site-packages/scipy/stats/_axis_nan_policy.py:592: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


Fold 1 | Lasso | Logistic Regression
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.6111, F1: 0.6052
Fold 1 | Lasso | Random Forest
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.6111, F1: 0.6374
Fold 1 | Lasso | SVM-Linear
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.5000, F1: 0.5251
Fold 1 | SVM-RFE | Logistic Regression
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.5000, F1: 0.5556
Fold 1 | SVM-RFE | Random Forest
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.5556, F1: 0.5519
Fold 1 | SVM-RFE | SVM-Linear
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.5556, F1: 0.5615
Fold 1 | Random Forest | Logistic Regression
  Train acc: 0.9881, F1: 0.9881
  Val   acc: 0.6111, F1: 0.6198
Fold 1 | Random Forest | Random Forest
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.5556, F1: 0.5615
Fold 1 | Random Forest | SVM-Linear
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.6111, F1: 0.6284
Fold 1 | SVM-RBF | Logistic Regression
  Train acc: 0.9881, F1: 0.9881
  Val   acc: 0.6667, F1: 0.6865
Fol

/Users/damianmoreno/Documents/DePaul_Cobab/GCA_Project/.venv/lib/python3.13/site-packages/scipy/stats/_axis_nan_policy.py:592: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


Fold 2 | Lasso | Logistic Regression
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.5000, F1: 0.4854
Fold 2 | Lasso | Random Forest
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.5556, F1: 0.5118
Fold 2 | Lasso | SVM-Linear
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.5556, F1: 0.5217
Fold 2 | SVM-RFE | Logistic Regression
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.6111, F1: 0.5952
Fold 2 | SVM-RFE | Random Forest
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.5556, F1: 0.5273
Fold 2 | SVM-RFE | SVM-Linear
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.5556, F1: 0.5417
Fold 2 | Random Forest | Logistic Regression
  Train acc: 0.9643, F1: 0.9642
  Val   acc: 0.5556, F1: 0.5293
Fold 2 | Random Forest | Random Forest
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.6111, F1: 0.5773
Fold 2 | Random Forest | SVM-Linear
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.5000, F1: 0.4617
Fold 2 | SVM-RBF | Logistic Regression
  Train acc: 0.9881, F1: 0.9881
  Val   acc: 0.5556, F1: 0.5141
Fol

...

Fold 3 | Lasso | Logistic Regression
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.4706, F1: 0.5131
Fold 3 | Lasso | Random Forest
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.4706, F1: 0.4611
Fold 3 | Lasso | SVM-Linear
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.4706, F1: 0.5125
Fold 3 | SVM-RFE | Logistic Regression
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.4706, F1: 0.5036
Fold 3 | SVM-RFE | Random Forest
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.5294, F1: 0.5415
Fold 3 | SVM-RFE | SVM-Linear
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.4706, F1: 0.5317
Fold 3 | Random Forest | Logistic Regression
  Train acc: 0.9881, F1: 0.9881
  Val   acc: 0.4706, F1: 0.4838
Fold 3 | Random Forest | Random Forest
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.6471, F1: 0.6500
Fold 3 | Random Forest | SVM-Linear
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.4118, F1: 0.4071
Fold 3 | SVM-RBF | Logistic Regression
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.4706, F1: 0.5030
Fol

/Users/damianmoreno/Documents/DePaul_Cobab/GCA_Project/.venv/lib/python3.13/site-packages/scipy/stats/_axis_nan_policy.py:592: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


Fold 4 | Lasso | Logistic Regression
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.7059, F1: 0.6861
Fold 4 | Lasso | Random Forest
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.7647, F1: 0.7083
Fold 4 | Lasso | SVM-Linear
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.5294, F1: 0.4875
Fold 4 | SVM-RFE | Logistic Regression
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.7059, F1: 0.6714
Fold 4 | SVM-RFE | Random Forest
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.7059, F1: 0.6714
Fold 4 | SVM-RFE | SVM-Linear
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.6471, F1: 0.5828
Fold 4 | Random Forest | Logistic Regression
  Train acc: 0.9875, F1: 0.9875
  Val   acc: 0.6471, F1: 0.6133
Fold 4 | Random Forest | Random Forest
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.7059, F1: 0.6209
Fold 4 | Random Forest | SVM-Linear
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.5294, F1: 0.4578
Fold 4 | SVM-RBF | Logistic Regression
  Train acc: 0.9750, F1: 0.9753
  Val   acc: 0.6471, F1: 0.5755
Fol

/Users/damianmoreno/Documents/DePaul_Cobab/GCA_Project/.venv/lib/python3.13/site-packages/scipy/stats/_axis_nan_policy.py:592: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


Fold 5 | Lasso | Logistic Regression
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.6471, F1: 0.6722
Fold 5 | Lasso | Random Forest
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.5882, F1: 0.6242
Fold 5 | Lasso | SVM-Linear
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.6471, F1: 0.6484
Fold 5 | SVM-RFE | Logistic Regression
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.7059, F1: 0.7189
Fold 5 | SVM-RFE | Random Forest
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.5882, F1: 0.5614
Fold 5 | SVM-RFE | SVM-Linear
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.6471, F1: 0.6475
Fold 5 | Random Forest | Logistic Regression
  Train acc: 0.9881, F1: 0.9881
  Val   acc: 0.7059, F1: 0.7222
Fold 5 | Random Forest | Random Forest
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.6471, F1: 0.6722
Fold 5 | Random Forest | SVM-Linear
  Train acc: 1.0000, F1: 1.0000
  Val   acc: 0.4706, F1: 0.4991
Fold 5 | SVM-RBF | Logistic Regression
  Train acc: 0.9881, F1: 0.9881
  Val   acc: 0.7059, F1: 0.6889
Fol